# Sample size for the A/B by merchant · ParcelCheck (P4)
DRAFT for Luli (tasks G, J) · `06_07_experiments_operations.md` §2 · `spec.md` §11. No formula to learn: three inputs (the smallest difference, the spread, the clustering) give the number of claims per arm.

Decision metric: **refund cost per damaged claim** (exists in both arms). Unit: **the merchant** (claims of one merchant are correlated → design effect).

In [ ]:
from statistics import NormalDist

import pandas as pd

ALPHA, POWER = 0.05, 0.80                 # 1 false win in 20 · 1 missed win in 5 (the convention)
Z = NormalDist().inv_cdf(1 - ALPHA / 2) + NormalDist().inv_cdf(POWER)
CLAIMS_PER_DAY = 2250 / 30                # 02_sizing.xlsx: 25,000 tickets × 9 % damaged ≈ 2,250 a month


def n_per_arm(delta_eur, sigma_eur=6.0, m=3, icc=0.05):
    """Claims per arm to see a difference of delta_eur in mean cost per claim, merchant-clustered."""
    n = 2 * Z**2 * sigma_eur**2 / delta_eur**2
    return n * (1 + (m - 1) * icc)


base = n_per_arm(1.0)
print(f"base case (δ = €1, σ = €6, m = 3, ICC = 0.05): {base:.0f} claims per arm · {2 * base:.0f} in total · "
      f"{2 * base / CLAIMS_PER_DAY:.1f} days of damaged claims")

## How the number moves with each input (the CFO question is the first column)

In [ ]:
rows = []
for delta in (0.5, 1.0, 2.0, 3.0):
    for icc in (0.01, 0.05, 0.10):
        n = n_per_arm(delta, icc=icc)
        rows.append({"smallest difference δ (€)": delta, "ICC": icc, "claims per arm": round(n),
                     "total": round(2 * n), "days": round(2 * n / CLAIMS_PER_DAY, 1)})
pd.DataFrame(rows)

In [ ]:
sigmas = [4.0, 6.0, 8.0]
pd.DataFrame({f"σ = €{s:.0f}": [round(n_per_arm(d, sigma_eur=s)) for d in (0.5, 1.0, 2.0, 3.0)] for s in sigmas},
             index=[f"δ = €{d}" for d in (0.5, 1.0, 2.0, 3.0)])

## Reading it
- **Halve the difference you want to see → four times the claims.** δ = €1 needs ≈ 622 claims per arm (≈ 17 days); δ = €0.5 needs ≈ 2,500 (≈ 2 months).
- We size for **€1** so the test can tell *kill* (< €1) from *iterate* (€1–3); the break-even of the module is **€3** per claim (`06_07` §2).
- **Duration is 4 weeks anyway**: the slowest guardrail (wrong refusals confirmed by contests within 7 days, leakage on the weekly 5 % sample) needs four weekly readings.
- σ and ICC are **guesses**: re-estimate them from the two shadow-mode weeks, then re-run this notebook before the A/B starts.